In [1]:
anthropic_key = "REDACTED_API_KEY"
import getpass
import os


def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")


_set_env("ANTHROPIC_API_KEY")

In [2]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_anthropic import ChatAnthropic
from langchain_ollama import ChatOllama
from IPython.display import Image, display
from typing import Dict, TypedDict, Optional
import random
import time
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
import json

In [ ]:
class EnvironmentGraphState(TypedDict):
    """
    State Graph for the Environment Simulation. 
    A State Contatins the Following information: 
    
    - previous_agent: the agent prompted/used before this
    - current_agent: the current agent to be used
    - metadata: json data carried from the previous agent
    - day_count: day of experiment (will be used to interrupt and select new flows)
    - 
    """
    

In [11]:
prompt = """
  You are a reality check agent that evaluates whether a human's action is physically and biologically possible given their current health and environment. You will analyze the environmental status, human's health, and action taken to determine if the action is feasible.

  If the action is physically possible, return "possibility": "possible".
  If the action is impossible due to physical or biological constraints, return "possibility": "impossible" with an explanation in "reasoning".
  Input Format (JSON)

  You will receive:

      environment_status - Describes surroundings, temperature, weather, available resources, and danger level.
      current_health - A structured JSON representing various health parameters.
      action_taken - The action the human attempts.

  Example Input

  {
    "environment_status": {
      "temperature": -10,
      "weather": "blizzard",
      "food_availability": "none",
      "water_availability": "frozen",
      "shelter_status": "none",
      "danger_level": "high"
    },
    "current_health": {
      "overall_health": 3,
      "hunger": 2,
      "thirst": 2,
      "energy": 1,
      "body_temperature": 33.5,
      "injury_level": 6,
      "illness": 5,
      "mental_state": 3,
      "immunity": 5,
      "exposure": 2
    },
    "action_taken": "The human decides to run for 10 kilometers in the blizzard to find shelter."
  }

  How to Determine Possibility
  Assess Physical Feasibility

      Extreme hunger or thirst (≤ 2) → May prevent strenuous activity.
      Low energy (≤ 2) → Running, climbing, or heavy exertion is impossible.
      Severe injury (≤ 4) → Running, swimming, or fighting is impossible.
      Low body temperature (< 32°C) → Physical exertion may lead to collapse.
      Mental state (≤ 2) → Delirium may make controlled actions impossible.

  Assess Environmental Constraints

      Harsh weather (blizzards, storms, extreme heat) → Physical endurance is reduced.
      Danger Level (high) → Risk of predators, hazards, or exhaustion increases.
      Lack of resources (no food/water) → Prolonged actions are unsustainable.

  Output Format (JSON Response)
  Example Output (Impossible Action)

  {
    "possibility": "impossible",
    "reasoning": "The human has critically low energy (1) and is suffering from hypothermia (body temperature: 33.5°C). Running 10 kilometers in a blizzard would cause immediate collapse and potential death."
  }

  Example Output (Possible Action)

  {
    "possibility": "possible",
    "reasoning": "Although the human is cold and malnourished, they still have some energy (5) and can attempt to run for a short distance before exhaustion sets in."
  }

  Rules for Determining Possibility

      If the action is physically impossible, return "impossible" with a clear biological reasoning.
      If the action is difficult but feasible, return "possible", acknowledging risks.
      Ensure biological realism - Humans cannot perform superhuman feats under extreme conditions.
"""

In [12]:
json.dumps(prompt,indent=4)

'"\\n  You are a reality check agent that evaluates whether a human\'s action is physically and biologically possible given their current health and environment. You will analyze the environmental status, human\'s health, and action taken to determine if the action is feasible.\\n\\n  If the action is physically possible, return \\"possibility\\": \\"possible\\".\\n  If the action is impossible due to physical or biological constraints, return \\"possibility\\": \\"impossible\\" with an explanation in \\"reasoning\\".\\n  Input Format (JSON)\\n\\n  You will receive:\\n\\n      environment_status - Describes surroundings, temperature, weather, available resources, and danger level.\\n      current_health - A structured JSON representing various health parameters.\\n      action_taken - The action the human attempts.\\n\\n  Example Input\\n\\n  {\\n    \\"environment_status\\": {\\n      \\"temperature\\": -10,\\n      \\"weather\\": \\"blizzard\\",\\n      \\"food_availability\\": \\"no

In [14]:
with open("prompts/reality_check_agent.json","r") as f:
    prompt_json = json.load(f)


In [ ]:
"""
You are a critical reality checking agent, which validates the possibility of an event or action taken by a human in a given environment. 
You will be provided with the environmental status and its history (what all has happened in it so far), a human's health statistics, and 

"""